# Hierarchical MARL Chip Placement - GPU Training on Google Colab

This notebook trains the hierarchical MARL chip placement models with GPU acceleration.

## 1. Setup & Dependencies

In [ ]:
# Check GPU availability
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# Clone the repository
import os
from pathlib import Path

REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO_NAME.git"  # Replace with actual repo URL
WORK_DIR = "/content/hierarchical-marl-chip-placement"

# Clone if not already present
if not Path(WORK_DIR).exists():
    !git clone {REPO_URL} {WORK_DIR}
    print(f"Repository cloned to {WORK_DIR}")
else:
    print(f"Repository already exists at {WORK_DIR}")

os.chdir(WORK_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Install dependencies
!pip install -q -e .
!pip install -q -r requirements.txt
print("Dependencies installed successfully!")

## 2. Configuration

In [ ]:
import yaml
from pathlib import Path

# Load and display config files
config_dir = Path("configs")

configs = {}
for config_file in ["env.yaml", "gnn.yaml", "rl.yaml", "training.yaml"]:
    config_path = config_dir / config_file
    if config_path.exists():
        with open(config_path, 'r') as f:
            configs[config_file.replace('.yaml', '')] = yaml.safe_load(f)
        print(f"\nLoaded {config_file}:")
        print(yaml.dump(configs[config_file.replace('.yaml', '')], default_flow_style=False))
    else:
        print(f"Warning: {config_file} not found")

## 3. Training Setup

In [ ]:
# Training parameters (customize as needed)
TRAINING_CONFIG = {
    "script": "scripts/train.py",  # Options: "scripts/train.py" or "scripts/train_gnn.py"
    "env_config": "configs/env.yaml",
    "gnn_config": "configs/gnn.yaml",
    "rl_config": "configs/rl.yaml",
    "training_config": "configs/training.yaml",
    "output_dir": "results/",
    "log_dir": "results/logs/",
    "checkpoint_dir": "checkpoints/",
}

print("Training Configuration:")
for key, value in TRAINING_CONFIG.items():
    print(f"  {key}: {value}")

In [ ]:
# Create output directories
from pathlib import Path

for dir_path in [TRAINING_CONFIG["output_dir"], 
                  TRAINING_CONFIG["log_dir"],
                  TRAINING_CONFIG["checkpoint_dir"]]:
    Path(dir_path).mkdir(parents=True, exist_ok=True)
    print(f"Created/verified directory: {dir_path}")

## 4. Run Training

In [ ]:
import subprocess
import sys

# Build training command
# Modify this based on your training script's expected arguments
cmd = [
    sys.executable,
    TRAINING_CONFIG["script"],
    "--env-config", TRAINING_CONFIG["env_config"],
    "--gnn-config", TRAINING_CONFIG["gnn_config"],
    "--rl-config", TRAINING_CONFIG["rl_config"],
    "--training-config", TRAINING_CONFIG["training_config"],
    "--output-dir", TRAINING_CONFIG["output_dir"],
    "--log-dir", TRAINING_CONFIG["log_dir"],
]

print(f"Running training command:")
print(" ".join(cmd))
print("\n" + "="*80 + "\n")

# Run training
result = subprocess.run(cmd, capture_output=False, text=True)

if result.returncode == 0:
    print("\n" + "="*80)
    print("Training completed successfully!")
else:
    print(f"\nTraining failed with return code: {result.returncode}")

## 5. Monitor Results

In [ ]:
from pathlib import Path
import json

# List checkpoint files
checkpoint_dir = Path(TRAINING_CONFIG["checkpoint_dir"])
if checkpoint_dir.exists():
    checkpoints = list(checkpoint_dir.glob("*.pt"))
    print(f"Found {len(checkpoints)} checkpoint files:")
    for ckpt in sorted(checkpoints):
        print(f"  - {ckpt.name}")
else:
    print("Checkpoint directory not found")

# List log files
log_dir = Path(TRAINING_CONFIG["log_dir"])
if log_dir.exists():
    logs = list(log_dir.glob("*.log")) + list(log_dir.glob("*.json"))
    print(f"\nFound {len(logs)} log files:")
    for log in sorted(logs):
        print(f"  - {log.name}")
else:
    print("\nLog directory not found")

In [ ]:
from google.colab import files
from pathlib import Path
import shutil

# Download results
results_dir = Path(TRAINING_CONFIG["output_dir"])
if results_dir.exists():
    # Create archive of results
    archive_path = shutil.make_archive("training_results", "zip", results_dir)
    print(f"Downloading results from {results_dir}...")
    files.download(archive_path)
    print("Download complete!")
else:
    print(f"Results directory {results_dir} not found")

## Troubleshooting

If you encounter issues:

1. **GPU not available**: Go to Runtime → Change runtime type and select GPU
2. **Import errors**: Check that the repository URL is correct and the project structure is as expected
3. **Missing config files**: Verify that config files exist in the `configs/` directory
4. **Training script arguments**: Modify the `cmd` list in cell 4 to match your script's expected arguments

### Alternative: Manual training command

You can also run training manually with custom arguments:

In [ ]:
# Uncomment and modify as needed for your specific training requirements
# !cd {WORK_DIR} && python scripts/train.py --help